# Pipeline for Transcribing Videos with WhisperX

02/20/2026 | 
Author: Cristian Ortega Singer | 
Mail: cris.ortega@fau.de


ChatAI by GWDG Gesellschaft für wissenschaftliche Datenverarbeitung was used for asistance in refining parts of the pipeline and for debbuging:

>Doosthosseini, Ali, Jonathan Decker, Hendrik Nolte, and Julian Kunkel. 2026. “SAIA: A Seamless Slurm-Native Solution for HPC-Based Services.” The Journal of Supercomputing 82 (7): 403. https://doi.org/10.1007/s11227-026-08508-3.

Install Dependencies

In [2]:
#%pip install -U whisperx pandas tqdm srt
!ffmpeg -version


ffmpeg version 8.0.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
built with gcc 15.2.0 (Rev8, Built by MSYS2 project)
configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdread --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopenjpeg --enable-libquirc --enable-libuavs3d --enable-libxevd --enable-libzvbi --enable-liboapv --enable-libqrencode --enable-librav1e --enable-libsvtav1 --enable-libvvenc --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxavs2 --enable-libxeve --enable-libxvid --enable-libaom --enable-libjxl --enable-libvpx --enab

Import dependencies

In [ ]:
from pathlib import Path
import pandas as pd
import re

import subprocess
import whisperx
import torch

import json
import pandas as pd
from pathlib import Path

Create data structure with files to work with

In [ ]:
BASE_DIR = Path("downloads")
WORK_AUDIO = Path("work/audio")
WORK_AUDIO.mkdir(parents=True, exist_ok=True)

TRANS_DIR = Path("transcripts")
TRANS_DIR.mkdir(exist_ok=True)

video_rows = []

for folder in BASE_DIR.iterdir():
    if not folder.is_dir():
        continue
    
    # match pattern: YYYY-MM-DD_VIDEOID
    m = re.match(r"(\d{4}-\d{2}-\d{2})_(.+)", folder.name)
    if not m:
        continue
    
    date, video_id = m.groups()
    
    # find video file
    video_files = list(folder.glob("*.mp4")) + list(folder.glob("*.mkv")) + list(folder.glob("*.webm"))
    if not video_files:
        continue
    
    video_path = video_files[0]  # assume one
    
    video_rows.append({
        "video_id": video_id,
        "date": date,
        "video_path": str(video_path),
        "folder": str(folder)
    })

manifest = pd.DataFrame(video_rows)
manifest.head(), len(manifest)


(      video_id        date                                         video_path  \
 0  wtsSeRrOBZQ  2012-11-15  downloads\2012-11-15_wtsSeRrOBZQ\2012-11-15_wt...   
 1  8HHUta7n0y4  2013-04-26  downloads\2013-04-26_8HHUta7n0y4\2013-04-26_8H...   
 2  bzkLtFLpWEg  2013-04-26  downloads\2013-04-26_bzkLtFLpWEg\2013-04-26_bz...   
 3  eDrIDaJRwJ0  2013-04-26  downloads\2013-04-26_eDrIDaJRwJ0\2013-04-26_eD...   
 4  OKz_pgqQycc  2013-04-26  downloads\2013-04-26_OKz_pgqQycc\2013-04-26_OK...   
 
                              folder  
 0  downloads\2012-11-15_wtsSeRrOBZQ  
 1  downloads\2013-04-26_8HHUta7n0y4  
 2  downloads\2013-04-26_bzkLtFLpWEg  
 3  downloads\2013-04-26_eDrIDaJRwJ0  
 4  downloads\2013-04-26_OKz_pgqQycc  ,
 2349)

Extract audio files from videos

In [ ]:
def extract_audio(video_path, audio_path):
    cmd = [
        "ffmpeg", "-y",
        "-i", str(video_path),
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        str(audio_path)
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)


manifest["audio_path"] = manifest["video_id"].apply(
    lambda vid: str(WORK_AUDIO / f"{vid}.wav")
)

for i, row in manifest.iterrows():
    ap = Path(row["audio_path"])
    if ap.exists():
        continue
    extract_audio(row["video_path"], ap)



Load and configure WhisperX model

In [ ]:
device = "cuda"
compute_type = "float16"
LANGUAGE = "de"

model = whisperx.load_model(
    "large-v3",
    device=device,
    compute_type=compute_type,
    language=LANGUAGE
)

align_model, align_metadata = whisperx.load_align_model(
    language_code=LANGUAGE,
    device=device
)


c:\Users\CrisO\anaconda3\envs\whisperx\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\CrisO\anaconda3\envs\whisperx\lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is 

2026-02-19 10:05:49 - whisperx.vads.pyannote - INFO - Performing voice activity detection using Pyannote...


Lightning automatically upgraded your loaded checkpoint from v1.5.4 to v2.6.1. To apply the upgrade to your files permanently, run `python -m lightning.pytorch.utilities.upgrade_checkpoint c:\Users\CrisO\anaconda3\envs\whisperx\lib\site-packages\whisperx\assets\pytorch_model.bin`
Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_voxpopuli_base_10k_asr_de.pt" to C:\Users\CrisO/.cache\torch\hub\checkpoints\wav2vec2_voxpopuli_base_10k_asr_de.pt
100%|██████████| 360M/360M [00:08<00:00, 47.0MB/s] 


Transcribe audio files with WhisperX

In [ ]:
errors = []

def transcribe_and_align(audio_path, video_id):
    result = model.transcribe(audio_path, batch_size=8)
    
    aligned = whisperx.align(
        result["segments"],
        align_model,
        align_metadata,
        audio_path,
        device
    )
    
    out_path = TRANS_DIR / f"{video_id}.whisperx.json"
    out_path.write_text(
        json.dumps(aligned, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )

for i, row in manifest.iterrows():
    print(i+1)
    out_json = TRANS_DIR / f"{row['video_id']}.whisperx.json"
    if out_json.exists():
        print("Skipping existing transcript: " + row["video_id"])
        continue
    try:
        transcribe_and_align(row["audio_path"], row["video_id"])
    except Exception as e:
        print("An Error occurred in: " + row["audio_path"] + " " + row["video_id"])
        print(e)
        errors.append(row)

fails = pd.DataFrame(errors)
fails.to_csv("fails.csv", index=False)


1
Skipping existing transcript: wtsSeRrOBZQ
2
Skipping existing transcript: 8HHUta7n0y4
3
Skipping existing transcript: bzkLtFLpWEg
4
Skipping existing transcript: eDrIDaJRwJ0
5
Skipping existing transcript: OKz_pgqQycc
6
Skipping existing transcript: r7exgls85kg
7
Skipping existing transcript: -Bf375yCUP8
8
Skipping existing transcript: -P2nVYjqaNc
9
Skipping existing transcript: 1K2NYb9Qg1o
10
Skipping existing transcript: MUbvifkMXK8
11
Skipping existing transcript: Nt7DeLYSzkk
12
Skipping existing transcript: Ub-vVIXU4kE
13
Skipping existing transcript: 3249HNziLYQ
14
Skipping existing transcript: fKMYi5MKaHw
15
Skipping existing transcript: i8aoKtzoeYk
16
Skipping existing transcript: IqBcwdvrpkY
17
Skipping existing transcript: mN71ltMREcw
18
Skipping existing transcript: _wQ51d8TFeI
19
Skipping existing transcript: bXohZYOu28o
20
Skipping existing transcript: dwJdLf1Qr0w
21
Skipping existing transcript: HOxg9DV6LNM
22
Skipping existing transcript: KLeW8ThmENQ
23
Skipping existin